# Módulo F5 · Herramientas del oficio — Práctica guiada

**Proyecto:** creas, versions y documentas un mini-proyecto **analisis-tiendas** desde cero, todo desde Python con celdas de bash (`%%bash`): la idea es ver de primera mano cómo encajan terminal, Git y entornos.

> Todo esto se ejecuta en un sandbox temporal dentro de `/tmp/f5-sandbox` para no tocar tu carpeta real de proyectos. Al final de la práctica, ese flujo lo puedes hacer en tu repositorio real.


In [ ]:
%%bash
# --- 5.1 · La terminal: sistema de ficheros ---------------------------------
# Dónde estamos y qué hay
pwd
echo "----"
ls -a

In [ ]:
%%bash
# --- 5.1 · Pipes y redirección ----------------------------------------------
# Creamos datos de ejemplo (CSV) para practicar
mkdir -p /tmp/f5-sandbox/analisis-tiendas
cd /tmp/f5-sandbox/analisis-tiendas
echo "id,fecha,importe,ciudad" > pedidos.csv
echo "1,2024-01-05,120,Madrid"  >> pedidos.csv
echo "2,2024-01-12,90,Barcelona" >> pedidos.csv
echo "3,2024-01-19,250,Madrid"  >> pedidos.csv
echo "4,2024-01-26,150,Valencia" >> pedidos.csv
echo "5,2024-02-02,80,Madrid"   >> pedidos.csv
echo "6,2024-02-09,300,Sevilla" >> pedidos.csv

cat pedidos.csv        # ver el archivo completo
echo "----"
grep "Madrid" pedidos.csv | wc -l      # pedidos de Madrid

In [ ]:
%%bash
# --- 5.2 · Git: iniciar el repo --------------------------------------------
cd /tmp/f5-sandbox/analisis-tiendas
git init -q 2>/dev/null || true
git config user.name  "Estudiante"   # en tu repo real: tu nombre
git config user.email "estudiante@ejemplo.com"

echo "pedidos.csv    datos brutos (no lo subas a GitHub)" > .gitignore
echo "app.py" >> /dev/null   # nada más

cat > app.py << 'EOF'
import csv

def pedidos_de(ciudad):
    with open("pedidos.csv") as f:
        return [r for r in csv.DictReader(f) if r["ciudad"] == ciudad]

if __name__ == "__main__":
    for row in pedidos_de("Madrid"):
        print(row)
EOF

cat > README.md << 'EOF'
# Analisis de TiendasSur

Mini-proyecto del modulo F5: analiza los pedidos de una cadena de tiendas.

## Uso

    python app.py              # pedidos de Madrid
    python -c "import app; print(app.pedidos_de('Valencia'))"

## Dependencias

Ninguna: solo la libreria estándar de Python.
EOF

git add .gitignore app.py README.md
git commit -q -m "Proyecto inicial: script, README y gitignore"
git log --oneline

**Qué acabas de hacer:** has creado un repo Git con un primer commit limpio (script + README + .gitignore).

En tu proyecto real, el flujo es el mismo: lo que aquí va en celdas va en tu terminal.


In [ ]:
%%bash
# --- 5.3 · Ramas y merge ---------------------------------------------------
cd /tmp/f5-sandbox/analisis-tiendas
git switch -c feature-filtro-ciudad     # rama para el cambio
# ... "trabajas": añades una mejora
cat > mejora.md << 'EOF'
# Mejora propuesta

- Filtro por ciudad con argumento en línea de comando.
EOF
git add mejora.md
git commit -q -m "Propuesta: filtro por ciudad por CLI"
git switch main                          # vuelve a lo estable
git merge -q feature-filtro-ciudad 2>/dev/null || git merge --no-edit feature-filtro-ciudad
git log --oneline

In [ ]:
%%bash
# --- 5.5 · Entorno virtual (venv) -------------------------------------------
cd /tmp/f5-sandbox/analisis-tiendas
python3 -m venv .venv 2>/dev/null || echo "(sandbox sin venv; en tu máquina funciona)"
if [ -d .venv ]; then
  .venv/bin/pip install -q pandas numpy 2>/dev/null
  .venv/bin/pip freeze > requirements.txt
  cat requirements.txt
  . .venv/bin/activate && python -c "import pandas; print('venv OK; pandas', pandas.__version__)"
else
  echo "En Colab/este notebook el venv no es necesario; en tu máquina:"
  echo "  python3 -m venv .venv && source .venv/bin/activate && pip install -r requirements.txt"
fi

## 5.6 · Notebooks que salvan: reproducibilidad

En este punto el flujo está claro. La última pieza es el hábito del notebook profesional:
1. **Reiniciar y ejecutar todo** (Restart & Run All) antes de entregas.
2. **Clear outputs** antes de commitear (los outputs se regeneran).
3. **Semillas fijas** (`default_rng(42)`).
4. **Secciones markdown** con la pregunta de negocio.
5. **Celda de conclusiones** al final.

Como este notebook también ejecuta `%%bash`, cumple la regla: se puede correr de arriba abajo.


---

## Resumen del notebook

- Terminal: moverse, ver, filtrar con pipes.
- Git: un flujo de 4 comandos (init/add/commit/log) cubre el 90 % del trabajo.
- Ramas para arriesgar; merge para traer; PR para colaborar.
- Un venv por proyecto; `requirements.txt` como contrato.
- El notebook profesional **se ejecuta de arriba abajo**.

---

## Ejercicios (5, entregables) — sobre un proyecto TUYO en GitHub

**1.** Crea un repo local con `git init`, un `README.md` que explique qué hace, `app.py` (o script de análisis) y `.gitignore` (excluye `.venv/`, `__pycache__/`, datos grandes). Haz al menos **3 commits con mensajes descriptivos** (qué cambió, por qué).

**2.** Crea y activa un venv de tu proyecto. Instala las dependencias (`pip install pandas numpy`, lo que uses) y genera `pip freeze > requirements.txt`. Revisa el archivo: ¿todas las líneas tienen sentido? ¿Hay alguna librería que no reconozcas? (si es una dependencia transitiva, es normal).

**3.** Crea una rama `feature/mejora`, añade una mejora real (función nueva, un gráfico, un ejemplo nuevo en el README) con un commit y merge a main con `git merge feature/mejora`. Pega el `git log --oneline` final.

**4.** Revisa el historial con `git log --oneline`. ¿Cuál de los commits está menos bien escrito? ¿Cómo lo reescribirías?

**5.** Publica tu repo en GitHub (o prepáralo para hacerlo con la instrucción de la próxima sesión) y entrega el enlace. Recuerda: **README + requirements + notebook con outputs limpios**.
